# Scrape AI Engineer Job Postings

In this notebook, you will scrape recent AI Engineer job postings and save them as the first dataset for the project.

This is step 1 of the AI Engineer Job Analysis workflow. The next notebooks will use the JSONL file you create here for classification, responsibility extraction, and skill extraction.

We use the Python scraping library [JobSpy](https://github.com/speedyapply/JobSpy) because it gives us a simple `scrape_jobs` function for collecting job postings from multiple job platforms.

In [ ]:
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

## Scrape Recent Job Postings

The next cell searches LinkedIn and Indeed for full-time AI Engineer jobs in the USA from the last 30 days.

If scraping takes too long on your machine, reduce `results_wanted` from `50` to a smaller value such as `10`.

In [ ]:
jobs = scrape_jobs(
    site_name=["linkedin", "indeed"],  # zip_recruiter, glassdoor, google, bayt, bdjobs
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=720,  # 30 days
    results_wanted=50,
)

print(f"Total jobs scraped: {len(jobs)}")

If scraping fails, first pull the latest version of the course repository.

After pulling the latest changes, refresh the project environment with:

`uv sync`

If JobSpy needs a newer version because a job platform changed, upgrade only that package:

`uv sync --upgrade-package python-jobspy`

You can still continue with the course if scraping does not work, because the repository includes example data in the `jobs` folder.

## Convert to a DataFrame

JobSpy returns tabular job data. We convert it to a pandas DataFrame so the filtering and deduplication steps are easier to write and inspect.

In [ ]:
jobs_df = pd.DataFrame(jobs)

## Keep Jobs With Required Fields

For the later LLM steps, each job needs a title, a URL, and a description.

This cell removes rows where one of those fields is missing or empty.

In [ ]:
# We keep only jobs that have the fields we want to store later.
jobs_before_required_filter = len(jobs_df)

required_columns = ["title", "job_url", "description"]
has_required_values = (
    jobs_df[required_columns]
    .fillna("")
    .apply(lambda column: column.astype(str).str.strip() != "")
    .all(axis=1)
)
jobs_df = jobs_df[has_required_values].copy()

print(f"Jobs with title, job URL, and description: {len(jobs_df)}")
print(
    f"Jobs filtered out because of missing required fields: {jobs_before_required_filter - len(jobs_df)}"
)

## Keep AI Engineer Titles

Indeed may return jobs where the phrase `AI Engineer` appears only in the description.

For this project, we want postings where the title itself contains both `AI` and `Engineer`, so the dataset stays focused on the role we are analyzing.

In [ ]:
# We ensure the title contains BOTH "AI" and "Engineer".
# We need to do this because, for Indeed, the job descriptions are also searched.
# That means we could get jobs that only have "AI Engineer" in the description.
jobs_before_title_filter = len(jobs_df)

title_contains_ai = jobs_df["title"].str.contains("AI", case=False, na=False)
title_contains_engineer = jobs_df["title"].str.contains(
    "Engineer", case=False, na=False
)
mask = title_contains_ai & title_contains_engineer
matching_jobs = jobs_df[mask].copy()

print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(
    f"Jobs filtered out because of the title filter: {jobs_before_title_filter - len(matching_jobs)}"
)

## Remove Duplicate Postings

The same role can appear more than once across job platforms.

This cell removes duplicate title and company combinations so each remaining row represents a unique posting for our analysis.

In [ ]:
# We drop duplicates based on the combination of title and company.
jobs_before_deduplication = len(matching_jobs)

filtered_jobs = matching_jobs.drop_duplicates(subset=["title", "company"]).copy()

print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")
print(
    f"Jobs filtered out as duplicates: {jobs_before_deduplication - len(filtered_jobs)}"
)

## Inspect the Scraped Jobs

Before saving the dataset we display the title, company, and URL for each posting.

Open a few URLs and skim the descriptions. This gives you a quick feeling for what companies mean when they ask for an AI Engineer.

In [ ]:
results_to_show = filtered_jobs[["title", "company", "job_url"]].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

## Save the Jobs as JSONL

The next notebooks need this data as an input file, so we save the filtered jobs to `jobs/1-scraped_jobs.jsonl`.

JSONL means JSON Lines: each line is one complete JSON object. That makes it a convenient format for datasets where we process one record at a time.

Reference: [jsonlines.org](https://jsonlines.org/)

In [ ]:
output_dir = Path("jobs")

jsonl_path = output_dir / "1-scraped_jobs.jsonl"
jobs_to_save = filtered_jobs[["title", "job_url", "description"]].copy()
jobs_to_save.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

print(f"Saved {len(jobs_to_save)} jobs to {jsonl_path}")